<a href="https://colab.research.google.com/github/jenil23/hbb-crispr-target-finder/blob/main/crispr_hbb_sickle_cell.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install biopython -q


In [ ]:
from Bio import Entrez , SeqIO

In [ ]:
Entrez.email = "jenil231109@gmail.com"

In [ ]:
handle = Entrez.efetch(db="nucleotide" , id="NM_000518", rettype="gb", retmode="text")

In [ ]:
record = SeqIO.read(handle, "genbank")

In [ ]:
handle.close()

In [ ]:
print(record.id)

NM_000518.5


In [ ]:
print(record.description)

Homo sapiens hemoglobin subunit beta (HBB), mRNA


In [ ]:
print(len(record.seq))

628


In [ ]:
for feature in record . features:
  if feature . type == "CDS" :
    cds = feature.extract(record.seq)
    break

In [ ]:
protein = cds.translate(to_stop=True)

In [ ]:
print(len(cds))

444


In [ ]:
print(len(protein))

147


In [ ]:
print(protein[:10])

MVHLTPEEKS


In [ ]:
codon_index = 6
start = codon_index * 3
codon = cds [start:start+3]

print(codon)
print(codon.translate())

GAG
E


In [ ]:
mutant_cds = cds[:start] + "GTG" + cds[start+3:]
mutant_protein = mutant_cds.translate(to_stop=True)

print(protein[:10])
print(mutant_protein[:10])

MVHLTPEEKS
MVHLTPVEKS


In [ ]:
import re

cds_offset = int(feature.location.start)
mut_pos = cds_offset + 19

seq = str(record.seq)

pam_positions = [m.start() for m in re.finditer("(?=[ACGT]GG)", seq)]

nearby = [p for p in pam_positions if abs(p - mut_pos) <= 30 and p >= 20]

for p in nearby:
  guide = seq[p-20:p]
  pam = seq[p:p+3]
  cut_distance = (p - 3) - mut_pos
  print(p, guide, pam, cut_distance)


51 GCAACCTCAAACAGACACCA TGG -21
69 CATGGTGCATCTGACTCCTG AGG -3
95 AGTCTGCCGTTACTGCCCTG TGG 23
96 GTCTGCCGTTACTGCCCTGT GGG 24
97 TCTGCCGTTACTGCCCTGTG GGG 25


In [ ]:
best = min(nearby, key=lambda p: abs((p - 3) - mut_pos))
print(best, seq[best-20:best], seq[best:best+3])

69 CATGGTGCATCTGACTCCTG AGG


In [ ]:
from Bio.SeqUtils import gc_fraction

for p in nearby:
    guide = seq[p-20:p]
    gc = gc_fraction(guide) * 100
    print(p, guide, round(gc, 1))

51 GCAACCTCAAACAGACACCA 50.0
69 CATGGTGCATCTGACTCCTG 55.0
95 AGTCTGCCGTTACTGCCCTG 60.0
96 GTCTGCCGTTACTGCCCTGT 60.0
97 TCTGCCGTTACTGCCCTGTG 60.0


In [ ]:
best = min(nearby, key=lambda p: abs((p - 3) - mut_pos))
guide = seq[best-20:best]
pam = seq[best:best+3]
gc = gc_fraction(guide) * 100
cut_distance = (best - 3) - mut_pos

print("Best guide:", guide)
print("PAM:", pam)
print("Cut distance from mutation:", cut_distance)
print("GC content:", round(gc, 1), "%")

Best guide: CATGGTGCATCTGACTCCTG
PAM: AGG
Cut distance from mutation: -3
GC content: 55.0 %
